In [1]:
import pandas as pd

df = pd.read_csv("../1_Extract/extracted_data.csv", index_col="date", parse_dates=True)

In [ ]:
# Cleaning
df = df.loc["2019-01-02":"2026-11-09"].copy()

# UKA fill: before UK ETS (pre 2021-05-19) UK was in EU ETS -> EUA in GBP (ratio = 1)
#           after (2022 gap) -> EUA in GBP x last observed UKA/EUA ratio
for uka, eua in [("UKA_M1", "EUA_Front"), ("UKA_M2", "EUA_2")]:
    eua_gbp = df[eua] * df["EUR/GBP"]
    ratio = (df[uka] / eua_gbp).ffill().fillna(1)
    df[uka] = df[uka].fillna(eua_gbp * ratio)

df = df.ffill(limit=3)                                                   # holidays: carry last price max 3 days
df = df.dropna()                                                         # drop anything still missing
df

,UK_BASE_M1,UK_BASE_M2,UK_BASE_M3,EUR/GBP,EUR/USD,NBP_M1,NBP_M2,NBP_M3,DE_BASE_M1,DE_BASE_M2,...,LS_Oil_M2,LS_Oil_M3,TTF_M1,TTF_M2,TTF_M3,UKA_M1,UKA_M2,COAL_M1,COAL_M2,COAL_M3
date,,,,,,,,,,,,,,,,,,,,,
2019-01-02,63.69,61.42,59.15,0.89968,1.1344,62.16,60.65,56.84,55.71,61.09,...,518.00,519.00,22.473,22.173,21.073,22.545981,22.617955,85.45,85.15,84.80
2019-01-03,63.11,60.53,58.34,0.90214,1.1394,61.81,60.25,56.44,55.42,60.13,...,512.50,513.00,22.256,21.971,20.900,20.821391,20.884541,83.75,83.25,82.90
2019-01-04,64.17,61.70,59.40,0.89541,1.1395,63.55,61.73,57.67,54.88,59.97,...,529.00,529.25,22.930,22.633,21.375,21.077951,21.131676,83.45,83.20,82.65
2019-01-07,60.84,58.64,57.45,0.89792,1.1474,59.26,58.04,54.92,53.73,58.39,...,543.50,543.50,21.531,21.331,20.504,19.835053,19.879949,81.50,80.95,80.50
2019-01-08,62.21,59.69,58.25,0.89974,1.1441,60.73,59.39,56.03,53.12,60.54,...,546.75,547.25,22.092,21.820,20.958,20.406103,20.451090,82.05,81.75,81.40
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-11-03,176.77,193.29,187.67,0.83435,1.1193,226.17,227.80,225.96,232.33,203.67,...,1350.00,1306.75,91.602,91.258,90.563,67.060000,67.190000,140.20,140.70,139.75
2026-11-04,175.53,187.55,184.00,0.83350,1.1228,218.40,219.75,217.82,233.68,202.66,...,1288.00,1247.75,88.082,87.796,87.043,67.030000,67.170000,138.30,138.05,136.95
2026-11-05,181.34,191.66,187.06,0.83419,1.1219,224.92,226.23,223.98,236.37,208.06,...,1284.50,1242.00,90.679,90.386,89.669,66.440000,66.580000,137.30,137.00,135.80


In [3]:
# Unit conversions
THERM_PER_MWH = 34.1214    # 1 MWh = 34.1214 therms  -> NBP p/therm × 0.341214 = £/MWh
MWH_PER_T_COAL = 6.978     # API2 coal: 6,000 kcal/kg = 6.978 MWh per tonne
MWH_PER_T_GASOIL = 11.86   # ICE low-sulphur gasoil: ~11.86 MWh per tonne

# Carbon
CPS = 18.0                 # UK Carbon Price Support, £/tCO2 (on top of UKA)


In [ ]:
fx_usd = df["EUR/USD"]
fx_gbp = df["EUR/GBP"]

data = pd.DataFrame(index=df.index)

data["DE_POWER"] = df["DE_BASE_M1"]
data["TTF_GAS"]  = df["TTF_M1"]

data["UK_POWER"] = df["UK_BASE_M1"] / fx_gbp
data["NBP_GAS"]  = df["NBP_M1"] * THERM_PER_MWH / 100 / fx_gbp

data["COAL"]     = df["COAL_M1"] / MWH_PER_T_COAL / fx_usd
data["GASOIL"]   = df["LS_Oil_M1"] / MWH_PER_T_GASOIL / fx_usd

data["EUA"]      = df["EUA_Front"]
data["UK_CO2"]   = (df["UKA_M1"] + CPS) / fx_gbp   # UKA + CPS

data.to_csv("transformed_data.csv")

In [ ]:
forwards = pd.DataFrame(index=df.index)

for m in ["M2", "M3"]:
    forwards[f"DE_POWER_{m}"] = df[f"DE_BASE_{m}"]
    forwards[f"TTF_GAS_{m}"]  = df[f"TTF_{m}"]

    forwards[f"UK_POWER_{m}"] = df[f"UK_BASE_{m}"] / fx_gbp
    forwards[f"NBP_GAS_{m}"]  = df[f"NBP_{m}"] * THERM_PER_MWH / 100 / fx_gbp

    forwards[f"COAL_{m}"]     = df[f"COAL_{m}"] / MWH_PER_T_COAL / fx_usd
    forwards[f"GASOIL_{m}"]   = df[f"LS_Oil_{m}"] / MWH_PER_T_GASOIL / fx_usd

forwards["EUA_M2"]    = df["EUA_2"]
forwards["UK_CO2_M2"] = (df["UKA_M2"] + CPS) / fx_gbp   # UKA + CPS

forwards.to_csv("transformed_forwards.csv")

,DE_POWER_M2,TTF_GAS_M2,UK_POWER_M2,NBP_GAS_M2,COAL_M2,GASOIL_M2,DE_POWER_M3,TTF_GAS_M3,UK_POWER_M3,NBP_GAS_M3,COAL_M3,GASOIL_M3,EUA_M2,UK_CO2_M2
date,,,,,,,,,,,,,,
2019-01-02,61.09,22.173,68.268718,23.002211,10.756908,38.501607,52.60,21.073,65.745598,21.557225,10.712693,38.575934,25.14,45.147114
2019-01-03,60.13,21.971,67.096016,22.788196,10.470732,37.925644,52.13,20.900,64.668455,21.347150,10.426711,37.962645,23.15,43.102557
2019-01-04,59.97,22.633,68.906981,23.523459,10.463525,39.143229,52.03,21.375,66.338325,21.976314,10.394355,39.161728,23.60,43.702523
2019-01-07,58.39,21.331,65.306486,22.055484,10.110463,39.939260,50.68,20.504,63.981201,20.869869,10.054259,39.939260,22.14,42.186329
2019-01-08,60.54,21.820,66.341388,22.522839,10.239832,40.293975,51.74,20.958,64.740925,21.248606,10.195991,40.330824,22.73,42.735779
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-11-03,203.67,91.258,231.665368,93.160603,18.014268,101.695697,220.30,90.563,224.929586,92.408121,17.892637,98.437668,90.67,102.103434
2026-11-04,202.66,87.796,225.014997,89.960140,17.619884,96.722780,215.36,87.043,220.755849,89.170046,17.479486,93.700193,90.91,102.183563
2026-11-05,208.06,90.386,229.755811,92.536285,17.499895,96.537327,219.42,89.669,224.241480,91.615953,17.346612,93.343216,89.69,101.391769
